# Attention in Transformers: Concepts and Code in PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

---
<a id='1'></a>
# Part 1: Self-Attention

Self-attention lets every token look at every other token and decide how much to attend to it.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

- **Q (Query):** what this token is looking for
- **K (Key):** what each token offers
- **V (Value):** the actual content each token contributes
- **√d_k:** scaling to prevent softmax saturation

In [ ]:
class SelfAttention(nn.Module):

    def __init__(self, d_model=2, row_dim=0, col_dim=1):
        ## d_model = number of embedding values per token
        ## default d_model=2 so we can verify math by hand
        ## in "Attention Is All You Need", d_model=512
        super().__init__()

        ## W_q, W_k, W_v: learned projection matrices, no bias per original paper
        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)

        self.row_dim = row_dim
        self.col_dim = col_dim

    def forward(self, token_encodings):
        ## Step 1: Project inputs into Q, K, V
        q = self.W_q(token_encodings)
        k = self.W_k(token_encodings)
        v = self.W_v(token_encodings)

        ## Step 2: Compute similarity scores: Q @ K^T
        sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))

        ## Step 3: Scale by sqrt(d_k)
        scaled_sims = sims / torch.tensor(k.size(self.col_dim) ** 0.5)

        ## Step 4: Softmax -> attention weights
        attention_percents = F.softmax(scaled_sims, dim=self.col_dim)

        ## Step 5: Weighted sum of values
        attention_scores = torch.matmul(attention_percents, v)

        return attention_scores

In [ ]:
## 3 tokens, each with a 2-dim embedding
encodings_matrix = torch.tensor([[1.16, 0.23],
                                  [0.57, 1.36],
                                  [4.41, -2.16]])

torch.manual_seed(42)

selfAttention = SelfAttention(d_model=2, row_dim=0, col_dim=1)

selfAttention(encodings_matrix)

In [ ]:
print("W_q:\n", selfAttention.W_q.weight.transpose(0, 1))
print("\nW_k:\n", selfAttention.W_k.weight.transpose(0, 1))
print("\nW_v:\n", selfAttention.W_v.weight.transpose(0, 1))

In [ ]:
q = selfAttention.W_q(encodings_matrix)
k = selfAttention.W_k(encodings_matrix)
v = selfAttention.W_v(encodings_matrix)

print("Q:\n", q)
print("\nK:\n", k)
print("\nV:\n", v)

In [ ]:
sims = torch.matmul(q, k.transpose(dim0=0, dim1=1))
print("Similarity scores (Q @ K^T):\n", sims)

In [ ]:
scaled_sims = sims / (torch.tensor(2) ** 0.5)
print("Scaled similarities:\n", scaled_sims)

In [ ]:
attention_percents = F.softmax(scaled_sims, dim=1)
print("Attention weights:\n", attention_percents)
print("\nRow sums (should all be 1.0):", attention_percents.sum(dim=1))

In [ ]:
output = torch.matmul(attention_percents, selfAttention.W_v(encodings_matrix))
print("Manual output:\n", output)
print("\nMatches forward pass?", torch.allclose(output, selfAttention(encodings_matrix), atol=1e-6))

---
<a id='2'></a>
# Part 2: Masked Self-Attention

Used in **decoder-only** models (GPT, LLaMA). Each token can only attend to itself and **previous** tokens — not future ones. This is called **causal masking**.

We set the upper triangle of the similarity matrix to `-inf` before softmax, so those positions become 0 after softmax.

```
Mask (3x3):
 0   -inf  -inf
 0    0    -inf
 0    0     0
```